# 📈 Thí Nghiệm Ablation Tốc Độ Hội Tụ Phase 2: LiDAR vs RS-LiDAR Trên Google Colab (GPU A100 / L4)

Notebook này thực hiện thí nghiệm **Ablation Tốc Độ Hội Tụ Phase 2 (Intermediate Convergence Dynamics)**:
- **Nguyên lý cốt lõi**: Chạy **đúng 1 lượt khử nhiễu duy nhất (Single Denoising Trajectory DDIM-50, $\eta=0.0$)**, giữ cố định 100% lịch trình timestep để cô lập hoàn toàn biến số guidance.
- **Trích xuất trạng thái trung gian**: Tại dải sweep các bước (ví dụ: `25, 30, 35, 40, 45, 50`), tự động trích xuất ước lượng ảnh sạch $\hat{x}_0^{(t)}$ (predicted original sample) từ DDIM scheduler, decode qua VAE và đo lường trực tiếp:
  - 🏆 **ImageReward** (Chỉ số guidance chính)
  - 📎 **CLIP-Score** (Độ bám sát text prompt)
  - 🌟 **HumanPreference (HPS v2.1)** (Thị hiếu thẩm mỹ người dùng)
  - 🎯 **GenEval Benchmark** (Mask2Former Swin-S & OpenCLIP ViT-L-14 chuẩn xác)
- **Kiểm chứng giả thuyết**: *Khi hàm reward được làm mịn bằng Randomized Smoothing, vector lái $\mathbf{v}_t$ trơn hơn, giúp bước nhảy rời rạc hóa DDIM chính xác hơn và hội tụ sớm hơn (đạt chất lượng cao ở ít bước hơn so với Vanilla LiDAR).*
- **Lưu trữ liên tục**: Toàn bộ ảnh trung gian, latents và bảng tổng kết được đồng bộ trực tiếp ra Google Drive sau mỗi prompt.


## 1. Kiểm Tra Phần Cứng GPU & VRAM
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> A100 (hoặc L4) GPU** và bật **High-RAM**.


In [ ]:
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_gb} GB")
    if vram_gb >= 22.0:
        print("✅ Tuyệt vời! GPU có đủ VRAM để kích hoạt Full-Batch VAE Decoding tốc độ cao.")
    else:
        print("⚠️ Chú ý: GPU dưới 22GB VRAM, code sẽ tự động dùng chunking để bảo vệ bộ nhớ.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")

!nvidia-smi


## 2. Gắn Kết Google Drive & Thiết Lập Lưu Trữ Tức Thì (Real-time Persistence)
Toàn bộ kết quả (ảnh tại từng checkpoint step và bảng hội tụ `convergence_summary.csv`) sẽ được ghi trực tiếp vào `My Drive/RS-LiDAR/` thông qua symlink.


In [ ]:
import os, shutil, subprocess
from google.colab import drive

# 1. Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
assert os.path.exists(base_drive), f"❌ Không tìm thấy thư mục Google Drive tại {base_drive}! Vui lòng gắn kết lại Drive."

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
os.makedirs(f"{DRIVE_DIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)

# 2. Clone hoặc kéo cập nhật repo RS-LiDAR tự động
REPO_DIR = "/content/RS-LiDAR"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/leekwanreal/RS-LiDAR.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/test_phase2_convergence_ablation.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}

# 3. Tạo Symbolic Links an toàn
for folder_name in ["Lookahead_samples", "Target_samples"]:
    link_path = os.path.join(WORKDIR, folder_name)
    drive_target = os.path.join(DRIVE_DIR, folder_name)
    os.makedirs(drive_target, exist_ok=True)

    if os.path.islink(link_path):
        if not os.path.exists(link_path):
            os.remove(link_path)
            os.symlink(drive_target, link_path)
    elif os.path.exists(link_path):
        print(f"📦 Phát hiện thư mục cục bộ {folder_name}, đang đồng bộ an toàn sang Google Drive...")
        for item in os.listdir(link_path):
            src_item = os.path.join(link_path, item)
            dst_item = os.path.join(drive_target, item)
            if not os.path.exists(dst_item):
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)
        shutil.rmtree(link_path)
        os.symlink(drive_target, link_path)
    else:
        os.symlink(drive_target, link_path)

print("✅ Hệ thống Symlink đã sẵn sàng 100%! Toàn bộ dữ liệu được bảo lưu vĩnh viễn trên Drive:")
print("  📁", DRIVE_DIR)


## 3. Cài Đặt Môi Trường Chuẩn Xác (100% Trơn Tru & Không Xung Đột)
Cài đặt bộ thư viện đồng bộ, tắt backend TensorFlow để chống xung đột Protobuf, và tải vocab HPSv2.


In [ ]:
import os, urllib.request

# 1. Tắt backend TensorFlow để loại bỏ xung đột Protobuf runtime_version
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# 2. Nâng cấp protobuf và cài đặt các thư viện tương thích PyTorch 2.x
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q image-reward || pip install -q --no-deps git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Đảm bảo file từ điển OpenCLIP cho HPSv2
import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    print("📥 Đang tải file BPE vocab cho HPSv2...")
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
print("✅ Môi trường thực nghiệm sẵn sàng 100%!")


## 4. Cấu Hình Siêu Tham Số Ablation Tốc Độ Hội Tụ & Dải Sweep Checkpoint Steps
Cấu hình phương pháp (`USE_SMOOTHING = True` cho RS-LiDAR hoặc `False` cho LiDAR) và dải sweep các bước trung gian (`EVAL_CHECKPOINTS`).


In [ ]:
import glob, os, json

# ==============================================================================
# 🎯 1. CHỌN MÔ HÌNH VÀ PHƯƠNG PHÁP GUIDANCE
# ==============================================================================
MODEL_CHOICE = "SD1.5"  # @param ["SD1.5", "SDXL-DPM-8"]

if MODEL_CHOICE == "SD1.5":
    MODEL_NAME_P1 = "runwayml/stable-diffusion-v1-5"
    MODEL_NAME_P2 = "runwayml/stable-diffusion-v1-5"
    MODEL_TAG = "SD15"
    SCALE = 12.5
    STEPS_P1 = 5
    NUM_PARTICLES_P1 = 50
elif MODEL_CHOICE == "SDXL-DPM-8":
    MODEL_NAME_P1 = "stabilityai/stable-diffusion-xl-base-1.0"
    MODEL_NAME_P2 = "stabilityai/stable-diffusion-xl-base-1.0"
    MODEL_TAG = "SDXL_DPM8"
    SCALE = 8.0
    STEPS_P1 = 8
    NUM_PARTICLES_P1 = 50
else:
    raise ValueError("Lựa chọn mô hình không hợp lệ!")

# ==============================================================================
# 🧪 2. CẤU HÌNH RANDOMIZED SMOOTHING (PHƯƠNG PHÁP GUIDANCE)
# - USE_SMOOTHING = True: Chạy RS-LiDAR (Smoothed Reward: sigma=1.0, M=4, CRN=True)
# - USE_SMOOTHING = False: Chạy Vanilla LiDAR (Unsmoothed Reward baseline)
# ==============================================================================
USE_SMOOTHING = True  # @param {type:"boolean"}
SIGMA = 1.0  # @param {type:"number"}
NUM_MC_SAMPLES = 4  # @param {type:"integer"}
USE_CRN = True  # @param {type:"boolean"}

# ==============================================================================
# 📊 3. CẤU HÌNH SWEEP CÁC BƯỚC ĐÁNH GIÁ TRUNG GIAN (INTERMEDIATE CHECKPOINT STEPS)
# - EVAL_CHECKPOINTS: Dải bước trung gian cần chụp ảnh x_0 và tính reward trong 1 run 50 bước
# - Ví dụ khuyến nghị: "25,30,35,40,45,50" hoặc "10,15,20,25,30,35,40,45,50"
# ==============================================================================
EVAL_CHECKPOINTS = "25,30,35,40,45,50"  # @param {type:"string"}

STEPS_P2 = 50          # Cố định DDIM 50 bước chuẩn Bảng 2
ETA_P2 = 0.0           # DDIM xác định (ODE) cô lập hoàn toàn quỹ đạo
SEED_P1 = 100
SEED_P2 = 100
LAMBDA = 5000.0
RESAMPLE_T_END = 200
NUM_PARTICLES_P2 = 4

# ==============================================================================
# 🎯 4. CẤU HÌNH TẬP PROMPTS:
# - "100_REPRESENTATIVE": 100 prompt phân tầng đủ 6 task GenEval (nhanh, chuẩn thống kê)
# - "ALL": Toàn bộ 553 prompts
# ==============================================================================
PROMPT_MODE = "100_REPRESENTATIVE"  # @param ["100_REPRESENTATIVE", "ALL", "FIRST_100"]

if PROMPT_MODE == "100_REPRESENTATIVE":
    MAX_PROMPT = 100
    PROMPT_INDICES_ARG = "stratified_100"
    SUBSET_TAG = "_100p"
elif PROMPT_MODE == "FIRST_100":
    MAX_PROMPT = 100
    PROMPT_INDICES_ARG = "0-99"
    SUBSET_TAG = "_first100p"
else:
    MAX_PROMPT = 553
    PROMPT_INDICES_ARG = None
    SUBSET_TAG = ""

OVERWRITE_PHASE1 = False
OVERWRITE_PHASE2 = False

is_sdxl = "xl" in MODEL_CHOICE.lower()
expected_dim = 128 if is_sdxl else 64

# ==============================================================================
# 🔍 5. TỰ ĐỘNG PHÁT HIỆN LOOKAHEAD ĐÃ HOÀN THÀNH (AUTO-ALIAS PHASE 1)
# ==============================================================================
step_variants = [f"Step{STEPS_P1}", f"DPM{STEPS_P1}", f"DPM{STEPS_P1}_Step{STEPS_P1}"]
lookahead_candidates_master = []

if USE_SMOOTHING:
    crn_variants = ["_crn", ""] if locals().get("USE_CRN", False) else ["", "_crn"]
    for s_var in step_variants:
        for c_var in crn_variants:
            lookahead_candidates_master.append(
                f"RSLookahead_{MODEL_TAG}_{s_var}_n{NUM_PARTICLES_P1}_sig{SIGMA}_M{NUM_MC_SAMPLES}{c_var}_seed{SEED_P1}"
            )
    LOOKAHEAD_MASTER = lookahead_candidates_master[0]
else:
    for s_var in step_variants:
        lookahead_candidates_master.append(
            f"Lookahead_{MODEL_TAG}_{s_var}_n{NUM_PARTICLES_P1}_seed{SEED_P1}"
        )
    lookahead_candidates_master.extend([
        f"{SEED_P1}_{NUM_PARTICLES_P1}_{STEPS_P1}",
        f"100_{NUM_PARTICLES_P1}_{STEPS_P1}"
    ])
    LOOKAHEAD_MASTER = lookahead_candidates_master[0]

LOOKAHEAD_TAG = f"{LOOKAHEAD_MASTER}{SUBSET_TAG}"

found_complete_tag = None
if not OVERWRITE_PHASE1:
    for cand in lookahead_candidates_master:
        cand_tag = f"{cand}{SUBSET_TAG}"
        check_paths = [
            f"{WORKDIR}/Lookahead_samples/{cand_tag}",
            f"{DRIVE_DIR}/Lookahead_samples/{cand_tag}" if 'DRIVE_DIR' in locals() and DRIVE_DIR else ""
        ]
        for p in check_paths:
            if p and os.path.exists(p):
                lat_files = glob.glob(f"{p}/[0-9]*/samples/latent.pt")
                if lat_files:
                    try:
                        import torch
                        chk = torch.load(lat_files[0], map_location="cpu")
                        chk_t = chk[0] if isinstance(chk, (list, tuple)) else chk
                        if hasattr(chk_t, "shape") and chk_t.shape[-1] != expected_dim:
                            continue
                    except Exception:
                        pass
                res_count = len(glob.glob(f"{p}/[0-9]*/results.json"))
                if res_count >= MAX_PROMPT:
                    found_complete_tag = cand_tag
                    break
        if found_complete_tag:
            break

if found_complete_tag:
    if found_complete_tag != LOOKAHEAD_TAG:
        print("=" * 70)
        print(f"⚡ AUTO-ALIAS: Đã tự động phát hiện thư mục Lookahead Phase 1 hoàn chỉnh ({MAX_PROMPT}/{MAX_PROMPT} prompts):")
        print(f"   📁 Thư mục tìm thấy: '{found_complete_tag}'")
        print(f"   -> Tự động alias LOOKAHEAD_TAG = '{found_complete_tag}' để SKIP hoàn toàn Phase 1!")
        print("=" * 70)
        LOOKAHEAD_TAG = found_complete_tag

lookahead_dir = f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}"

# Tên thư mục thực nghiệm Phase 2 Convergence Ablation
method_label = "RSLiDAR" if USE_SMOOTHING else "LiDAR"
crn_tag = "_crn" if (USE_SMOOTHING and USE_CRN) else ""
RUN_NAME = f"Convergence_{method_label}_{MODEL_TAG}_DDIM{STEPS_P2}_s{SCALE}_lmbda{int(LAMBDA)}{crn_tag}{SUBSET_TAG}"
target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"

existing_look = len(glob.glob(f"{lookahead_dir}/[0-9]*/results.json")) if not OVERWRITE_PHASE1 else 0
existing_targ = len(glob.glob(f"{target_dir}/step_{STEPS_P2}/[0-9]*/results.json")) if os.path.exists(target_dir) else 0

print(f"📊 TIẾN ĐỘ THỰC NGHIỆM ABLATION HỘI TỤ (MÔ HÌNH {MODEL_TAG}):")
print(f"  🔹 Phương pháp:            {method_label} ({'Smoothed σ=' + str(SIGMA) + ', M=' + str(NUM_MC_SAMPLES) if USE_SMOOTHING else 'Vanilla'})")
print(f"  🔹 Dải Checkpoint Sweep:   {EVAL_CHECKPOINTS}")
print(f"  📁 Thư mục Phase 1:        {LOOKAHEAD_TAG} ({existing_look}/{MAX_PROMPT} prompts)")
print(f"  📁 Thư mục Target Phase 2: {RUN_NAME} ({existing_targ}/{MAX_PROMPT} prompts)")
print(f"  ----------------------------------------------------------------------")
if existing_look >= MAX_PROMPT:
    print("  ✅ Phase 1 đã hoàn thành đầy đủ! TỰ ĐỘNG SKIP Phase 1.")
else:
    print("  ⚠️ Cần chạy Phase 1 cho Lookahead trước khi sang Phase 2.")


## 5. Phase 1: Lấy Mẫu Lookahead & Đánh Giá Reward (Lookahead Sampling)
Nếu Phase 1 đã hoàn thành và lưu trên Google Drive, cell này sẽ **TỰ ĐỘNG SKIP trong 0.00 giây**.


In [ ]:
import os, glob
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
%cd {WORKDIR}

existing_look = len(glob.glob(f"{lookahead_dir}/[0-9]*/results.json")) if os.path.exists(lookahead_dir) else 0

if existing_look >= MAX_PROMPT and not OVERWRITE_PHASE1:
    print("=" * 70)
    print(f"✅ TỰ ĐỘNG SKIP PHASE 1: Thư mục '{LOOKAHEAD_TAG}' đã có đủ {existing_look}/{MAX_PROMPT} prompts!")
    print("=" * 70)
else:
    cmd = [
        "python", "lookahead_sampling.py",
        f"--seed={SEED_P1}",
        f"--num_particles={NUM_PARTICLES_P1}",
        f"--num_inference_steps={STEPS_P1}",
        f"--model_name={MODEL_NAME_P1}",
        "--guidance_reward_fn=ImageReward",
        "--metrics_to_compute=ImageReward",
        "--prompt_path=prompt_files/geneval_metadata.jsonl",
        f"--max_prompt={MAX_PROMPT}",
        f"--run_name={LOOKAHEAD_TAG}",
        "--resume"
    ]
    if 'PROMPT_INDICES_ARG' in locals() and PROMPT_INDICES_ARG:
        cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")
    if USE_SMOOTHING:
        cmd.extend(["--use_smoothing", f"--sigma={SIGMA}", f"--num_mc_samples={NUM_MC_SAMPLES}"])
        if locals().get("USE_CRN", False):
            cmd.append("--use_crn")
    if OVERWRITE_PHASE1:
        cmd.append("--overwrite")

    phase_desc = "RS-LiDAR Lookahead" if USE_SMOOTHING else "Vanilla Lookahead"
    print(f"🚀 Đang khởi chạy Phase 1 ({phase_desc}):")
    print(" ".join(cmd))
    print("-" * 70)
    !{" ".join(cmd)}


## 5.1. Kiểm Tra Tính Toàn Vẹn Script Convergence Ablation
Kiểm tra cú pháp của file độc lập `test_phase2_convergence_ablation.py` trước khi thực hiện Phase 2.


In [ ]:
import os, py_compile

filepath = "test_phase2_convergence_ablation.py"
if not os.path.exists(filepath):
    filepath = f"{WORKDIR}/test_phase2_convergence_ablation.py"

with open(filepath, "r", encoding="utf-8") as f:
    code_content = f.read()

# 1. Đảm bảo bản vá Step 50 gán final_images vô điều kiện
old_guard = "if args.num_inference_steps not in pipe._checkpoint_predictions:\n                pipe._checkpoint_predictions[args.num_inference_steps] = final_images"
new_assignment = "pipe._checkpoint_predictions[args.num_inference_steps] = final_images"
if old_guard in code_content:
    code_content = code_content.replace(old_guard, new_assignment)

# 2. Đảm bảo hook bỏ qua decode Tweedie tại terminal step
if "current_step == pipe._total_inference_steps:" not in code_content:
    hook_target = "if current_step in pipe._eval_checkpoint_steps:"
    hook_patch = "if current_step in pipe._eval_checkpoint_steps:\n            if pipe._total_inference_steps is not None and current_step == pipe._total_inference_steps:\n                return step_output"
    code_content = code_content.replace(hook_target, hook_patch, 1)

# 3. Đảm bảo loại bỏ upcast_vae trong decode_latents_to_pil để VAE giữ nguyên FP16 chuẩn LiDAR
if "pipe.upcast_vae()" in code_content:
    old_upcast = """        needs_upcasting = (
            hasattr(pipe.vae, "config")
            and getattr(pipe.vae.config, "force_upcast", False)
            and pipe.vae.dtype == torch.float16
        )
        lat = latents.clone()
        if needs_upcasting and hasattr(pipe, "upcast_vae"):
            pipe.upcast_vae()
            lat = lat.to(next(iter(pipe.vae.post_quant_conv.parameters())).dtype)
        elif lat.dtype != pipe.vae.dtype:
            lat = lat.to(pipe.vae.dtype)"""
    new_fp16 = """        lat = latents.clone()
        if lat.dtype != pipe.vae.dtype:
            lat = lat.to(pipe.vae.dtype)"""
    code_content = code_content.replace(old_upcast, new_fp16)

with open(filepath, "w", encoding="utf-8") as f:
    f.write(code_content)

try:
    py_compile.compile(filepath, doraise=True)
    print(f"✅ File {filepath} hoàn toàn chuẩn cú pháp và đã tích hợp đầy đủ 100% bản vá!")
except Exception as e:
    raise RuntimeError(f"❌ Lỗi cú pháp trong {filepath}: {e}")


## 6. Phase 2: Lấy Mẫu Đích & Đánh Giá Tốc Độ Hội Tụ Trung Gian (Convergence Ablation)
Chạy bộ giải DDIM 50 bước ($\eta=0.0$). Tại mỗi bước trong dải sweep `EVAL_CHECKPOINTS` (ví dụ `25, 30, 35, 40, 45, 50`), script tự động decode $\hat{x}_0$ và tính **ImageReward, CLIP-Score, HPS v2.1**.


In [ ]:
import os, glob
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
%cd {WORKDIR}

cmd = [
    "python", "test_phase2_convergence_ablation.py",
    f"--seed={SEED_P2}",
    f"--model_name={MODEL_NAME_P2}",
    f"--num_particles={NUM_PARTICLES_P2}",
    f"--num_inference_steps={STEPS_P2}",
    f"--eval_checkpoint_steps={EVAL_CHECKPOINTS}",
    f"--eta={ETA_P2}",
    "--use_rag",
    f"--lookahead_path={LOOKAHEAD_TAG}",
    f"--top_k={NUM_PARTICLES_P1}",
    f"--scale={SCALE}",
    f"--lmbda={LAMBDA}",
    f"--resample_t_end={RESAMPLE_T_END}",
    "--prompt_path=prompt_files/geneval_metadata.jsonl",
    f"--max_prompt={MAX_PROMPT}",
    "--guidance_reward_fn=ImageReward",
    "--metrics_to_compute=ImageReward#Clip-Score#HumanPreference",
    "--save_individual_images",
    f"--run_name={RUN_NAME}",
    "--resume"
]

if 'PROMPT_INDICES_ARG' in locals() and PROMPT_INDICES_ARG:
    cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")

if OVERWRITE_PHASE2:
    cmd.append("--overwrite")

print(f"🚀 Đang khởi chạy Phase 2 Convergence Ablation ({method_label}):")
print(" " + "-" * 70)
print(" ".join(cmd))
print("-" * 70)
!{" ".join(cmd)}


## 7. Chấm Điểm Benchmark GenEval Qua Từng Checkpoint Step (Mask2Former Swin-S & CLIP)
Đánh giá độ chính xác thực thể và không gian của GenEval tại mỗi checkpoint step (`step_25`, `step_30`, ..., `step_50`) để quan sát sự hình thành đối tượng.


In [ ]:
# ==============================================================================
# 🎯 GENEVAL BENCHMARK EVALUATION ACROSS ALL INTERMEDIATE CHECKPOINTS
# ==============================================================================
import os, glob, json, time, urllib.request
import numpy as np, pandas as pd
from PIL import Image
import torch
from tqdm import tqdm
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

device = "cuda" if torch.cuda.is_available() else "cpu"

metadata_path = f"{WORKDIR}/prompt_files/geneval_metadata.jsonl"
if not os.path.exists(metadata_path):
    os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/leekwanreal/RS-LiDAR/main/prompt_files/geneval_metadata.jsonl", metadata_path)

prompts_meta = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            prompts_meta.append(json.loads(line.strip()))

# Danh sách các bước cần đánh giá GenEval
checkpoint_list = [int(s.strip()) for s in EVAL_CHECKPOINTS.split(",") if s.strip().isdigit()]
checkpoint_list.sort()

# Danh sách 80 lớp COCO chuẩn bài báo GenEval (object_names.txt)
COCO_CLASSES_80 = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "computer mouse", "tv remote",
    "computer keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book",
    "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

# Nạp mô hình Mask2Former và OpenCLIP chuẩn 100% Table 2
print("📦 Đang nạp mô hình Mask2Former Swin-S COCO và OpenCLIP ViT-L-14...")
mask2former_id = "facebook/mask2former-swin-small-coco-instance"
image_processor = AutoImageProcessor.from_pretrained(mask2former_id)
image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}
detector = Mask2FormerForUniversalSegmentation.from_pretrained(mask2former_id).to(device).eval()
id2label = detector.config.id2label

import open_clip
clip_arch = "ViT-L-14"
clip_model, _, clip_transform = open_clip.create_model_and_transforms(clip_arch, pretrained="openai", device=device)
clip_tokenizer = open_clip.get_tokenizer(clip_arch)
clip_model.eval()

COLORS = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink', 'brown', 'black', 'white']
COLOR_CLASSIFIERS = {}

def get_color_classifier(classname):
    if classname not in COLOR_CLASSIFIERS:
        c_name = classname if classname else "object"
        templates = [
            f"a photo of a {c} {c_name}",
            f"a photo of a {c}-colored {c_name}",
            f"a photo of a {c} object"
        ]
        with torch.inference_mode():
            zeroshot_weights = []
            for c in COLORS:
                prompts = [t.format(c=c) for t in templates]
                tokens = clip_tokenizer(prompts).to(device)
                text_embeddings = clip_model.encode_text(tokens)
                text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)
                class_embedding = text_embeddings.mean(dim=0)
                class_embedding = class_embedding / class_embedding.norm(dim=-1, keepdim=True)
                zeroshot_weights.append(class_embedding)
            COLOR_CLASSIFIERS[classname] = torch.stack(zeroshot_weights, dim=1)
    return COLOR_CLASSIFIERS[classname]

def color_classification(image, objects, classname):
    if not objects:
        return []
    clf = get_color_classifier(classname)
    blank = Image.new("RGB", image.size, color="#999")
    crop_tensors = []
    for obj in objects:
        box, mask = obj[0], obj[1] if len(obj) > 1 else None
        x1, y1, x2, y2 = int(max(0, box[0])), int(max(0, box[1])), int(min(image.width, box[2])), int(min(image.height, box[3]))
        if (x2 - x1 <= 12 or y2 - y1 <= 12):
            crop_tensors.append(clip_transform(Image.new("RGB", (224, 224), color="#999")))
            continue
        comp = Image.composite(image, blank, Image.fromarray((mask > 0).astype(np.uint8) * 255)) if mask is not None else image
        crop_tensors.append(clip_transform(comp.crop((x1, y1, x2, y2))))
    
    crop_batch = torch.stack(crop_tensors).to(device)
    with torch.inference_mode():
        img_features = clip_model.encode_image(crop_batch)
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)
        logits = 100.0 * (img_features @ clf)
        return [COLORS[idx] for idx in logits.argmax(dim=-1).cpu().tolist()]

def relative_position(obj_a, obj_b):
    box_a = obj_a[0] if isinstance(obj_a, (tuple, list)) else obj_a
    box_b = obj_b[0] if isinstance(obj_b, (tuple, list)) else obj_b
    boxes = np.array([box_a[:4], box_b[:4]])[:, :4].reshape(2, 2, 2)
    center_a, center_b = boxes.mean(axis=-2)
    dim_a, dim_b = np.abs(np.diff(boxes, axis=-2))[..., 0, :]
    offset = center_a - center_b
    revised_offset = np.maximum(np.abs(offset) - 0.1 * (dim_a + dim_b), 0) * np.sign(offset)
    if np.all(np.abs(revised_offset) < 1e-3):
        return set()
    norm_offset = np.linalg.norm(offset)
    if norm_offset <= 1e-5:
        return set()
    dx, dy = revised_offset / norm_offset
    relations = set()
    if dx < -0.5: relations.add("left of")
    if dx > 0.5: relations.add("right of")
    if dy < -0.5: relations.add("above")
    if dy > 0.5: relations.add("below")
    return relations

def evaluate_single_image(image, objects, metadata):
    correct = True
    matched_groups = []
    for req in metadata.get('include', []):
        classname = req['class'].lower()
        matched = True
        found_objects = objects.get(classname, [])[:req['count']]
        if len(found_objects) < req['count']:
            correct = matched = False
        else:
            if 'color' in req:
                colors = color_classification(image, found_objects, classname)
                if colors.count(req['color'].lower()) < req['count']:
                    correct = matched = False
            if 'position' in req and matched:
                expected_rel, target_group = req['position']
                if target_group >= len(matched_groups) or matched_groups[target_group] is None:
                    correct = matched = False
                else:
                    for obj in found_objects:
                        for target_obj in matched_groups[target_group]:
                            true_rels = relative_position(obj, target_obj)
                            if expected_rel not in true_rels:
                                correct = matched = False
                                break
                        if not matched:
                            break
        if matched:
            matched_groups.append(found_objects)
        else:
            matched_groups.append(None)
    for req in metadata.get('exclude', []):
        classname = req['class'].lower()
        if len(objects.get(classname, [])) >= req['count']:
            correct = False
    return correct

def evaluate_step_folder(step_folder):
    sample_folders = sorted(glob.glob(f"{step_folder}/[0-9]*"))
    if not sample_folders:
        return 0.0
    task_results = {'single_object': [], 'two_object': [], 'counting': [], 'colors': [], 'position': [], 'color_attr': []}
    
    for s_dir in tqdm(sample_folders, desc=f"Chấm điểm {os.path.basename(step_folder)}", leave=False):
        p_name = os.path.basename(s_dir)
        if not p_name.isdigit():
            continue
        p_idx = int(p_name)
        if p_idx >= len(prompts_meta):
            continue
        meta = prompts_meta[p_idx]
        tag = meta.get('tag', 'single_object')
        if tag not in task_results:
            continue
            
        img_files = sorted(glob.glob(f"{s_dir}/samples/*.png")) or \
                    sorted([img for img in glob.glob(f"{s_dir}/*.png") if not img.endswith("grid.png")])
        if not img_files:
            continue
            
        conf_thr = 0.9 if tag == 'counting' else 0.3
        prompt_scores = []
        for img_path in img_files:
            try:
                img = Image.open(img_path).convert('RGB')
                image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}
                inputs = image_processor(images=img, return_tensors="pt").to(device)
                with torch.inference_mode():
                    outputs = detector(**inputs)

                results = image_processor.post_process_instance_segmentation(
                    outputs, target_sizes=[img.size[::-1]], threshold=conf_thr
                )[0]
                segmentation = results["segmentation"].detach().cpu().numpy()
                segments_info = sorted(results["segments_info"], key=lambda s: s.get("score", 0.0), reverse=True)

                detected = {}
                for seg in segments_info:
                    lbl_id = seg["label_id"]
                    c_name = COCO_CLASSES_80[lbl_id].lower() if (0 <= lbl_id < len(COCO_CLASSES_80)) else id2label.get(lbl_id, "unknown").lower()
                    mask = (segmentation == seg["id"])
                    y_indices, x_indices = np.where(mask)
                    if len(x_indices) > 0 and len(y_indices) > 0:
                        x1, x2 = float(np.min(x_indices)), float(np.max(x_indices))
                        y1, y2 = float(np.min(y_indices)), float(np.max(y_indices))
                        box = [x1, y1, x2, y2, seg.get("score", 1.0)]
                        if len(detected.setdefault(c_name, [])) < 16:
                            detected[c_name].append((box, mask))

                success = evaluate_single_image(img, detected, meta)
                prompt_scores.append(1.0 if success else 0.0)
            except Exception:
                pass
                
        if prompt_scores:
            task_results[tag].append(np.mean(prompt_scores))
            
    all_means = [np.mean(scores) for scores in task_results.values() if len(scores) > 0]
    overall_geneval = float(np.mean(all_means)) if all_means else 0.0
    return overall_geneval

geneval_convergence = []
print(f"📊 Đang chấm điểm GenEval cho các checkpoint: {checkpoint_list}")
for step_k in checkpoint_list:
    step_dir = f"{target_dir}/step_{step_k}"
    csv_file = f"{step_dir}/geneval_summary.csv"
    if os.path.exists(csv_file):
        try:
            df_exist = pd.read_csv(csv_file)
            if "overall_geneval" in df_exist.columns:
                ov_val = float(df_exist.iloc[0]['overall_geneval'])
            elif "geneval" in df_exist.columns:
                ov_val = float(df_exist.iloc[0]['geneval'])
            else:
                ov_val = float(df_exist.iloc[0, 1])
            print(f"  • Step {step_k:2d}: GenEval = {ov_val:.4f} (đã có từ cache)")
        except Exception:
            ov_val = evaluate_step_folder(step_dir)
            df_out = pd.DataFrame([{"step": step_k, "overall_geneval": ov_val}])
            df_out.to_csv(csv_file, index=False)
            print(f"  • Step {step_k:2d}: GenEval = {ov_val:.4f}")
    else:
        ov_val = evaluate_step_folder(step_dir)
        df_out = pd.DataFrame([{"step": step_k, "overall_geneval": ov_val}])
        df_out.to_csv(csv_file, index=False)
        print(f"  • Step {step_k:2d}: GenEval = {ov_val:.4f}")
    geneval_convergence.append({"step": step_k, "geneval": ov_val})
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_gen_conv = pd.DataFrame(geneval_convergence)
df_gen_conv.to_csv(f"{target_dir}/geneval_convergence_summary.csv", index=False)

if 'DRIVE_DIR' in locals() and DRIVE_DIR and os.path.exists(DRIVE_DIR):
    drive_target_dir = f"{DRIVE_DIR}/Target_samples/{os.path.basename(target_dir)}"
    os.makedirs(drive_target_dir, exist_ok=True)
    df_gen_conv.to_csv(f"{drive_target_dir}/geneval_convergence_summary.csv", index=False)

print("✅ Đã hoàn tất đánh giá GenEval across checkpoints!")


## 8. Bảng Tổng Hợp & Đồ Thị Đường Cong Hội Tụ (Convergence Curves Analysis)
Trực quan hóa đồ thị so sánh tốc độ hội tụ của **LiDAR** vs **RS-LiDAR** qua từng bước khử nhiễu.


In [ ]:
# @title 📊 BƯỚC 8: VẼ ĐỒ THỊ ĐƯỜNG CONG HỘI TỤ (CONVERGENCE CURVES)
import os, glob, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

# Tìm các thư mục thực nghiệm Convergence trên Drive/Workdir
folders_to_check = {
    "RS-LiDAR (Smoothed)": f"{WORKDIR}/Target_samples/Convergence_RSLiDAR_{MODEL_TAG}_DDIM{STEPS_P2}_s{SCALE}_lmbda{int(LAMBDA)}_crn{SUBSET_TAG}",
    "LiDAR (Vanilla)": f"{WORKDIR}/Target_samples/Convergence_LiDAR_{MODEL_TAG}_DDIM{STEPS_P2}_s{SCALE}_lmbda{int(LAMBDA)}{SUBSET_TAG}"
}
if 'DRIVE_DIR' in locals() and DRIVE_DIR:
    folders_to_check["RS-LiDAR (Drive)"] = f"{DRIVE_DIR}/Target_samples/Convergence_RSLiDAR_{MODEL_TAG}_DDIM{STEPS_P2}_s{SCALE}_lmbda{int(LAMBDA)}_crn{SUBSET_TAG}"
    folders_to_check["LiDAR (Drive)"] = f"{DRIVE_DIR}/Target_samples/Convergence_LiDAR_{MODEL_TAG}_DDIM{STEPS_P2}_s{SCALE}_lmbda{int(LAMBDA)}{SUBSET_TAG}"

dfs = {}
for label, fpath in folders_to_check.items():
    csv_path = os.path.join(fpath, "convergence_summary.csv")
    if os.path.exists(csv_path):
        df_tmp = pd.read_csv(csv_path)
        # Nạp thêm GenEval nếu có
        gen_csv = os.path.join(fpath, "geneval_convergence_summary.csv")
        if os.path.exists(gen_csv):
            df_gen = pd.read_csv(gen_csv)
            df_tmp = pd.merge(df_tmp, df_gen, on="step", how="left")
        clean_name = "RS-LiDAR (Smoothed)" if "RS" in label else "LiDAR (Vanilla)"
        dfs[clean_name] = df_tmp

# Nếu chỉ có thư mục hiện tại
if not dfs and os.path.exists(f"{target_dir}/convergence_summary.csv"):
    df_curr = pd.read_csv(f"{target_dir}/convergence_summary.csv")
    gen_csv = f"{target_dir}/geneval_convergence_summary.csv"
    if os.path.exists(gen_csv):
        df_gen = pd.read_csv(gen_csv)
        df_curr = pd.merge(df_curr, df_gen, on="step", how="left")
    dfs[method_label] = df_curr

if dfs:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    metrics_to_plot = [
        ("ImageReward_mean", "ImageReward (Guidance Alignment)", "ImageReward Score"),
        ("Clip-Score_mean", "CLIP-Score (Text Fidelity)", "CLIP Score"),
        ("HumanPreference_mean", "HPS v2.1 (Aesthetic Quality)", "HPS v2.1"),
        ("geneval", "GenEval Benchmark Accuracy", "Overall Score (0-1)")
    ]

    colors = {"RS-LiDAR (Smoothed)": "#1f77b4", "LiDAR (Vanilla)": "#d62728", "RSLiDAR": "#1f77b4", "LiDAR": "#d62728"}
    markers = {"RS-LiDAR (Smoothed)": "o", "LiDAR (Vanilla)": "s", "RSLiDAR": "o", "LiDAR": "s"}

    for idx, (col, title, ylabel) in enumerate(metrics_to_plot):
        ax = axes[idx]
        for name, df in dfs.items():
            if col in df.columns and not df[col].isna().all():
                c = colors.get(name, "#2ca02c")
                m = markers.get(name, "o")
                ax.plot(df["step"], df[col], label=name, color=c, marker=m, linewidth=2.5, markersize=8)
        ax.set_title(title, fontweight="bold", fontsize=12)
        ax.set_xlabel("Denoising Step (DDIM-50)", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(loc="best", frameon=True)
        ax.grid(True, linestyle="--", alpha=0.6)

    plt.suptitle(f"🚀 So Sánh Tốc Độ Hội Tụ: LiDAR vs RS-LiDAR (Mô Hình {MODEL_TAG}, {MAX_PROMPT} Prompts)", fontsize=15, fontweight="bold", y=1.03)
    plt.tight_layout()
    plot_save_path = f"{target_dir}/convergence_comparison_plot.png"
    plt.savefig(plot_save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Đã lưu biểu đồ phân tích hội tụ tại: {plot_save_path}")

    # Bảng số liệu chi tiết
    print("\n📋 BẢNG SỐ LIỆU ĐỐI CHIẾU HỘI TỤ THEO STEP:")
    for name, df in dfs.items():
        print(f"\n--- {name} ---")
        display_cols = [c for c in ["step", "ImageReward_mean", "Clip-Score_mean", "HumanPreference_mean", "geneval"] if c in df.columns]
        print(df[display_cols].to_string(index=False))
else:
    print("⚠️ Chưa tìm thấy file convergence_summary.csv nào để vẽ đồ thị.")


## 9. Trực Quan Hóa Quá Trình Kết Tinh Ảnh $\hat{x}_0$ Theo Bước Khử Nhiễu
Trình bày lưới ảnh thể hiện sự hình thành chi tiết từ bước 25 đến bước 50 của các prompt tiêu biểu.


In [ ]:
import glob
from PIL import Image
from IPython.display import display

checkpoint_list = [int(s.strip()) for s in EVAL_CHECKPOINTS.split(",") if s.strip().isdigit()]
checkpoint_list.sort()

# Lấy 3 prompt đầu tiên để so sánh sự tiến hóa qua các bước
sample_prompts = [0, 5, 11]

for p_idx in sample_prompts:
    step_imgs = []
    titles = []
    for step_k in checkpoint_list:
        img_candidates = glob.glob(f"{target_dir}/step_{step_k}/{p_idx:05d}/samples/*.png")
        if img_candidates:
            step_imgs.append(Image.open(img_candidates[0]))
            titles.append(f"Step {step_k}")
    
    if step_imgs:
        total_w = sum(img.width for img in step_imgs)
        max_h = max(img.height for img in step_imgs)
        combined = Image.new("RGB", (total_w, max_h))
        cur_x = 0
        for img in step_imgs:
            combined.paste(img, (cur_x, 0))
            cur_x += img.width
        print(f"🖼️ Tiến trình khử nhiễu Prompt #{p_idx:05d} ({' -> '.join(titles)}):")
        display(combined.resize((min(1024, total_w // 2), max_h // 2)))


## 10. (Tùy Chọn) Tự Động Ngắt Kết Nối Máy Ảo Để Tiết Kiệm Compute Units
Bỏ dấu `#` ở 2 dòng dưới đây nếu bạn chạy qua đêm và muốn Colab tự động dừng tính Compute Units khi hoàn thành.


In [ ]:
# from google.colab import runtime
# runtime.unassign()
print("🎉 Hoàn tất toàn bộ quy trình thí nghiệm Phase 2 Convergence Ablation!")
